# Out-of-fold FinBERT sentiment scoring (expanding window)

Rebuilds the headline sentiment feature so that **every headline is scored by a model that never
saw it**, replacing the single checkpoint that was fine-tuned on 2019-01 → 2024-03 and then applied
to those same headlines.


## Fold layout

| slice | period | scored by a model trained on |
|---|---|---|
| 0 | 2019-01 → 2020-01 | *nothing available* → left as `NaN` |
| 1 | 2020-01 → 2021-01 | 2019-01 → 2020-01 − gap |
| 2 | 2021-01 → 2022-01 | 2019-01 → 2021-01 − gap |
| 3 | 2022-01 → 2023-01 | 2019-01 → 2022-01 − gap |
| 4 | 2023-01 → 2024-01 | 2019-01 → 2023-01 − gap |
| 5 | 2024-01 → 2024-06 | 2019-01 → 2024-01 − gap |
| 6 | 2024-06 → 2025-06 | 2019-01 → 2024-06 − gap |

Slice 6 covers the whole downstream forecasting/backtest evaluation window with a model fitted and
selected entirely on data before **2024-06**: the sentiment model is fixed before June 2024 and then
generates June 2024 – June 2025. Splitting 2024-01 → 2024-06 off as its own slice keeps that
boundary clean while still scoring the early-2024 headlines out of fold.

## Purge gap

`GAP = 1 day`, rather than the 4 h the label horizon alone would suggest. The composite label window straddles publication
(T−2 h → T+2 h15 winter, T−1 h → T+3 h15 summer), so avoiding overlap between a training headline's
label window and a scored headline's label window needs ≥ 5 h 15 min. A day is cheap (a handful of
headlines per boundary) and also absorbs the ≤3 h timezone ambiguity in the stored timestamps.

The same gap is applied between the train part and the validation part inside each fold.

## Model selection

Each fold holds out the **last 10 % of its own training window** (chronologically, behind the gap)
for early stopping. Nothing after the fold boundary is ever looked at. This closes the second
leakage path: the original runs watched the 2024-03 → 2025-03 evaluation window every epoch.

## Variants

Three config switches, everything else held identical, each combination writing to its own files so
runs stay comparable.

**`HEAD_MODE`**
- `tone3` *(default, published)* — FinBERT-tone's 3 logits → a `Linear(3, 1)` that is randomly
  initialised and *frozen*. Its realised weights are `[0.441, 0.479, −0.135]` against
  `id2label = {0: Neutral, 1: Positive, 2: Negative}`, i.e. Neutral and Positive score almost
  identically and Negative is barely penalised.
- `cls768` — trained `Linear(768, 1)` on the `[CLS]` vector. **Tested and found to be a wash**:
  test-window r moved 0.0325 → 0.0323 against the training target, well inside the ±0.019 standard
  error, while in-sample fold correlations *rose* (fold 1 0.34 → 0.38). Extra head capacity buys
  memorisation, not signal, so the published head is kept as the default.

**`TARGET_MODE` / `OUTPUT_ACT`** — the published combination is a badly conditioned pairing:

| quantity | measured at initialisation |
|---|---|
| tone logits | sd 5.47, range [−10.6, 12.3] |
| pre-`tanh` value | mean +0.99, sd 1.71; 36 % of headlines above \|2\|, 12 % above \|3\| |
| post-`tanh` output | mean +0.469, **sd 0.698**, saturating at ±1.000 |
| mean `tanh` gradient | **0.293** (1.0 = no saturation) |
| target `volume_surge_price` | **sd 0.0577** |

The model starts 12 standard deviations of target away from where it needs to be, and has to get
there through a nonlinearity whose gradient is attenuated ~3.4× on average and far more in the
tails — using only the last two encoder layers, since the head is frozen.

`TARGET_MODE = "zscore"` standardises the label with **that fold's training-window mean and standard
deviation only**, and `OUTPUT_ACT = "linear"` drops the squashing. The two must change together: a
standardised target has sd 1 and a `tanh` output cannot represent it. As a side effect the published
`HUBER_DELTA = 1.0` becomes meaningful — on the raw target (\|y\| < 0.25) the linear branch is
unreachable, so the "Adaptive Weighted Huber Loss" was exactly MSE.

In [ ]:
from pathlib import Path
import copy, json, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from transformers import BertModel, BertForSequenceClassification, BertTokenizerFast

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

# ------------------------------------------------------------------ paths
REPO    = Path.cwd().parent          # repository root
LABELS  = REPO / "weights_comparison_and_derive" / "output.csv"
CLEANED = REPO / "backtesting" / "data" / "cleaned"
OLD_CSV = CLEANED / "predicted_sentiments_finbert.csv"   # single-checkpoint column, for comparison

OUT = Path.cwd() / "out"
OUT.mkdir(exist_ok=True)
CKPT_DIR = Path.cwd() / "oof_checkpoints"
CKPT_DIR.mkdir(exist_ok=True)

# ------------------------------------------------------------------ fold layout
# The 2024-06-01 boundary keeps the downstream evaluation window clean: the model
# that scores June 2024 onward is fitted and selected entirely on data before June 2024.
EDGES = ["2019-01-01", "2020-01-01", "2021-01-01", "2022-01-01",
         "2023-01-01", "2024-01-01", "2024-06-01", "2025-07-01"]
EDGES = [pd.Timestamp(e) for e in EDGES]
GAP   = pd.Timedelta(days=1)     # purge / embargo around every split
VAL_FRAC = 0.10                  # chronological tail of each train window, held out behind GAP

# ------------------------------------------------------------------ model / training
MODEL_NAME  = "yiyanghkust/finbert-tone"
MAX_LEN     = 128
UNFREEZE    = 2          # last N encoder layers, as in the original sandbox
SEED        = 42

HEAD_MODE   = "tone3"    # "tone3" (as in the sandbox: frozen random Linear(3,1)) | "cls768" (trained 768->1)
TARGET_MODE = "zscore"   # "raw" (as in the sandbox) | "zscore" (per-fold, train-window statistics only)
OUTPUT_ACT  = "linear"   # "tanh" (as in the sandbox) | "linear"
DROPOUT     = 0.1        # cls768 only

_PUBLISHED  = (TARGET_MODE == "raw" and OUTPUT_ACT == "tanh")
RUN_TAG     = {"cls768": "cls", "tone3": "tone3"}[HEAD_MODE] + ("" if _PUBLISHED else "_cond")

# Measured on this RTX 3060 (BERT-base, len 128, last-2 trainable):
#   bs=9  fp32 170 samp/s | bs=32 fp32 190 | bs=32 amp 471 | bs=64 amp 490 | infer amp 665
# Throughput plateaus at bs=64 (~4 GB peak). The card is compute-bound, not memory-bound, so
# raising the batch further burns VRAM without going faster — it only reduces the step count.
BATCH_SIZE  = 64
INFER_BATCH = 256
USE_AMP     = True
BASE_LR     = 7.35e-5    # original, tuned at bs=9
LR          = BASE_LR * (BATCH_SIZE / 9) ** 0.5   # sqrt scaling -> 1.96e-4 at bs=64
HEAD_LR     = 1e-3       # cls768 only: the head is randomly initialised and needs a faster rate
WEIGHT_DECAY = 0.094     # original
# delta=1 on the raw target (|y| < 0.25) never leaves the quadratic branch, i.e. it is exactly MSE.
# On a standardised target (sd 1) the published delta=1 becomes a genuine robust threshold.
HUBER_DELTA  = 1.0
MAX_EPOCHS  = 100        # original ran a flat 100; early stopping normally fires far sooner
PATIENCE    = 5

FIDELITY_MODE = False    # True -> reproduce the original loop exactly (bs=9, fp32, 100 epochs,
                         # no early stopping). ~5x slower and re-introduces the overfitting.
if FIDELITY_MODE:
    BATCH_SIZE, USE_AMP, LR, MAX_EPOCHS, PATIENCE = 9, False, BASE_LR, 100, MAX_EPOCHS

SAVE_CHECKPOINTS = True  # ~440 MB per fold

# ------------------------------------------------------------------ output files (per HEAD_MODE)
OOF_CSV = CLEANED / f"predicted_sentiments_finbert_oof_{RUN_TAG}.csv"
Z_CSV   = CLEANED / f"predicted_sentiments_finbert_oof_{RUN_TAG}_z.csv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
print(f"head={HEAD_MODE} target={TARGET_MODE} out={OUTPUT_ACT} delta={HUBER_DELTA} tag={RUN_TAG}")
print(f"batch={BATCH_SIZE} amp={USE_AMP} lr={LR:.3g} head_lr={HEAD_LR:.3g} "
      f"max_epochs={MAX_EPOCHS} patience={PATIENCE}")

## Data

Same filtering as the original sandbox: drop rows where either label component is missing
(20 258 → 20 251). Row order of `output.csv` is preserved so the resulting CSV is a drop-in
replacement for `predicted_sentiments_finbert.csv`.

In [ ]:
raw = pd.read_csv(LABELS)
df  = raw[raw["price_change_4h"].notna() & raw["volume_surge_price"].notna()].copy()
df["dt"] = pd.to_datetime(df["timestamp"], unit="s")
df = df.sort_values("dt", kind="mergesort")          # stable: keeps file order within a timestamp

print(f"rows: {len(raw)} -> {len(df)} after dropping missing labels")
print(f"span: {df['dt'].min()}  ->  {df['dt'].max()}")
y_all = df["volume_surge_price"]
print(f"label volume_surge_price: sd {y_all.std():.4f}, |q99| {y_all.abs().quantile(.99):.3f}, "
      f"range [{y_all.min():.2f}, {y_all.max():.2f}]")

df["slice"] = np.searchsorted(np.array(EDGES[1:], dtype="datetime64[ns]"),
                              df["dt"].values.astype("datetime64[ns]"), side="right")
print()
print(df.groupby("slice").agg(n=("dt", "size"), start=("dt", "min"), end=("dt", "max")).to_string())

## Model

`transformers` 5.x cannot resolve this checkpoint through `AutoModel*` (its `config.json` has no
`model_type` key), so the Bert classes are named explicitly.

**`cls768`** drops FinBERT-tone's tone classifier and the pooler entirely, taking
`last_hidden_state[:, 0]` — the raw `[CLS]` vector — through dropout into a trained `Linear(768, 1)`,
then `tanh`. Trainable: the last `UNFREEZE` encoder layers plus the head, with the head on its own
faster learning rate since it starts from scratch.

**`tone3`** is the published architecture, kept for comparison. Its frozen random head must be
*identical across folds* or the five slices would land on different scales and signs;
`torch.manual_seed(SEED)` before every construction guarantees this.

The output nonlinearity follows `OUTPUT_ACT`, so both heads can be run squashed or linear.

In [ ]:
def _squash(x):
    return torch.tanh(x) if OUTPUT_ACT == "tanh" else x


class FinBERTRegressionTone3(nn.Module):
    """Original sandbox architecture: 3 tone logits -> frozen random Linear(3, 1) -> tanh."""
    def __init__(self, model_name=MODEL_NAME):
        super().__init__()
        self.bert = BertForSequenceClassification.from_pretrained(model_name, num_labels=3)
        self.regressor = nn.Linear(3, 1)

    @property
    def encoder_layers(self):
        return self.bert.bert.encoder.layer

    def backbone_params(self):
        return self.bert.parameters()

    def forward(self, input_ids, attention_mask):
        logits = self.bert(input_ids=input_ids, attention_mask=attention_mask).logits
        return _squash(self.regressor(logits))


class FinBERTRegressionCLS(nn.Module):
    """[CLS] (768) -> dropout -> trained Linear(768, 1) -> tanh."""
    def __init__(self, model_name=MODEL_NAME):
        super().__init__()
        self.bert = BertModel.from_pretrained(model_name, add_pooling_layer=False)
        self.dropout = nn.Dropout(DROPOUT)
        self.regressor = nn.Linear(self.bert.config.hidden_size, 1)

    @property
    def encoder_layers(self):
        return self.bert.encoder.layer

    def backbone_params(self):
        return self.bert.parameters()

    def forward(self, input_ids, attention_mask):
        h = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0]
        return _squash(self.regressor(self.dropout(h)))


def build_model():
    torch.manual_seed(SEED)                 # identical head initialisation on every fold
    m = FinBERTRegressionCLS() if HEAD_MODE == "cls768" else FinBERTRegressionTone3()
    for p in m.backbone_params():
        p.requires_grad = False
    for layer in m.encoder_layers[-UNFREEZE:]:
        for p in layer.parameters():
            p.requires_grad = True
    for p in m.regressor.parameters():
        p.requires_grad = (HEAD_MODE == "cls768")   # tone3 keeps the head frozen, as published
    return m.to(device)


def param_groups(model):
    enc = [p for l in model.encoder_layers[-UNFREEZE:] for p in l.parameters() if p.requires_grad]
    head = [p for p in model.regressor.parameters() if p.requires_grad]
    groups = [{"params": enc, "lr": LR}]
    if head:
        groups.append({"params": head, "lr": HEAD_LR})
    return groups


_probe = build_model()
print(f"trainable {sum(p.numel() for p in _probe.parameters() if p.requires_grad)/1e6:.2f} M "
      f"of {sum(p.numel() for p in _probe.parameters())/1e6:.1f} M")
print("head:", tuple(_probe.regressor.weight.shape),
      "trainable" if _probe.regressor.weight.requires_grad else "FROZEN at random init")
del _probe; torch.cuda.empty_cache()

## Tokenise once, keep everything resident on the GPU

The original tokenises inside `Dataset.__getitem__`, which pins the CPU and starves the card. All
20 251 headlines padded to 128 tokens are 41 MB as `int64` — small enough to sit in VRAM for the
whole run, so batching becomes a tensor slice and the GPU never waits on the host.

In [ ]:
tok = BertTokenizerFast.from_pretrained(MODEL_NAME)
enc = tok(df["title"].astype(str).tolist(), truncation=True,
          padding="max_length", max_length=MAX_LEN, return_tensors="pt")

IDS  = enc["input_ids"].to(device)
MASK = enc["attention_mask"].to(device)
Y    = torch.tensor(df["volume_surge_price"].to_numpy(np.float32), device=device)
DT   = df["dt"].to_numpy("datetime64[ns]")

print(f"{IDS.shape[0]} headlines tokenised -> {(IDS.nbytes + MASK.nbytes)/1e6:.0f} MB on device")

In [ ]:
class AdaptiveWeightedHuberLoss(nn.Module):
    """Verbatim from the sandbox: plain Huber unless per-sample weights are passed.

    With delta=1 and |y| < 0.25 the linear branch is unreachable, so as configured in the published
    runs this is exactly MSE. Kept as-is here for comparability.
    """
    def __init__(self, delta=1.0):
        super().__init__()
        self.delta = delta

    def forward(self, predictions, targets, weights=None):
        error = predictions - targets
        abs_error = torch.abs(error)
        quadratic = torch.minimum(abs_error, torch.tensor(self.delta, device=error.device))
        linear = abs_error - quadratic
        loss = 0.5 * quadratic ** 2 + self.delta * linear
        if weights is not None:
            loss = loss * weights
        return loss.mean()


def _idx(lo=None, hi=None):
    """Positional indices of headlines with lo <= dt < hi."""
    m = np.ones(len(DT), bool)
    if lo is not None: m &= DT >= np.datetime64(lo)
    if hi is not None: m &= DT <  np.datetime64(hi)
    return np.flatnonzero(m)


@torch.no_grad()
def predict(model, idx, batch=INFER_BATCH):
    model.eval()
    out = torch.empty(len(idx), device=device)
    ix = torch.as_tensor(idx, device=device)
    for s in range(0, len(idx), batch):
        b = ix[s:s + batch]
        with torch.amp.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            out[s:s + batch] = model(IDS[b], MASK[b]).squeeze(-1).float()
    return out


def train_fold(tr_idx, va_idx, tag):
    """Train from scratch on tr_idx, early-stop on va_idx, return the best-val model."""
    model = build_model()
    opt = torch.optim.AdamW(param_groups(model), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = AdaptiveWeightedHuberLoss(delta=HUBER_DELTA)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    g = torch.Generator(device=device); g.manual_seed(SEED)

    tr = torch.as_tensor(tr_idx, device=device)
    va = torch.as_tensor(va_idx, device=device)

    # Target standardisation uses this fold's TRAIN window only — never the validation slice and
    # never the scored slice, so the statistic is available at scoring time.
    if TARGET_MODE == "zscore":
        mu_y, sd_y = Y[tr].mean().item(), Y[tr].std().item()
    else:
        mu_y, sd_y = 0.0, 1.0
    Yt = (Y - mu_y) / sd_y

    best = (np.inf, -1, None)
    t0 = time.time()

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        perm = tr[torch.randperm(len(tr), generator=g, device=device)]
        tot = 0.0
        for s in range(0, len(perm), BATCH_SIZE):
            b = perm[s:s + BATCH_SIZE]
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
                loss = loss_fn(model(IDS[b], MASK[b]).squeeze(-1), Yt[b])
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            tot += loss.item() * len(b)

        val_loss = loss_fn(predict(model, va_idx), Yt[va]).item()
        if val_loss < best[0] - 1e-9:
            best = (val_loss, epoch, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()})
            flag = " *"
        else:
            flag = ""
        if epoch % 5 == 0 or flag or epoch == 1:
            print(f"    [{tag}] epoch {epoch:3d}  train {tot/len(perm):.5f}  val {val_loss:.5f}{flag}")

        if not FIDELITY_MODE and epoch - best[1] >= PATIENCE:
            print(f"    [{tag}] early stop at {epoch} (best epoch {best[1]})")
            break

    model.load_state_dict(best[2])
    return model, {"best_epoch": best[1], "val_loss": best[0], "epochs_run": epoch,
                   "y_mu": round(mu_y, 5), "y_sd": round(sd_y, 5),
                   "train_sec": round(time.time() - t0, 1)}

## The expanding-window run

For slice *k* starting at `E[k]`:

- **train window** = everything before `E[k] − GAP`
- of that, the chronological last 10 % (behind another `GAP`) is the validation set
- the fitted model scores slice *k* and nothing else

In [ ]:
scores = np.full(len(df), np.nan, dtype=np.float64)
scored_by = np.array(["unavailable"] * len(df), dtype=object)
rows = []

for k in range(1, len(EDGES) - 1):
    lo, hi = EDGES[k], EDGES[k + 1]
    tag = f"fold{k}"
    target = _idx(lo, hi)
    if len(target) == 0:
        print(f"{tag}: no headlines in {lo.date()} -> {hi.date()}, skipped"); continue

    pool = _idx(None, lo - GAP)                       # purged train pool
    n_val = max(1, int(round(len(pool) * VAL_FRAC)))
    va = pool[-n_val:]
    val_start = DT[va[0]]
    tr = pool[DT[pool] < val_start - GAP.to_timedelta64()]   # purge again between train and val

    print(f"\n{tag}: score {lo.date()} -> {hi.date()}  n={len(target)}"
          f"   | train n={len(tr)} (to {str(DT[tr[-1]])[:10]})"
          f"   val n={len(va)} ({str(DT[va[0]])[:10]} -> {str(DT[va[-1]])[:10]})")

    model, info = train_fold(tr, va, tag)
    p = predict(model, target).cpu().numpy().astype(np.float64)
    scores[target] = p
    scored_by[target] = tag

    y = df["volume_surge_price"].to_numpy()
    r    = float(np.corrcoef(p, y[target])[0, 1])
    r_tr = float(np.corrcoef(predict(model, tr).cpu().numpy(), y[tr])[0, 1])
    rows.append(dict(fold=tag, start=str(lo.date()), end=str(hi.date()), n_scored=len(target),
                     n_train=len(tr), n_val=len(va), **info,
                     corr_oof=round(r, 4), corr_insample_train=round(r_tr, 4),
                     sd_oof=round(float(p.std()), 5)))
    print(f"    -> corr(score, volume_surge_price) out-of-fold {r:+.4f}  |  in-sample on its own "
          f"train set {r_tr:+.4f}  |  sd {p.std():.5f}  |  {info['train_sec']/60:.1f} min")

    if SAVE_CHECKPOINTS:
        torch.save(model.state_dict(), CKPT_DIR / f"finbert_oof_{RUN_TAG}_{tag}.pth")
    del model; torch.cuda.empty_cache()

folds = pd.DataFrame(rows)
print()
print(folds.to_string(index=False))
print(f"\ntotal train time: {folds['train_sec'].sum()/60:.1f} min")
folds.to_csv(OUT / f"oof_fold_summary_{RUN_TAG}.csv", index=False)

## Fold-scale calibration (required)

Out-of-fold scoring introduces a problem the single-checkpoint version did not have: **each fold is
a different model with a different output scale.** In the `tone3` run the raw slice standard
deviations ran 0.027 → 0.016 → 0.019 → 0.011 → 0.005, with ±0.02 level jumps at every January.
Concatenating raw slices would hand the iTransformer a feature with a step change at each fold
boundary and a variance that decays across the sample — an artefact worse than the one being
removed.

Fix: standardise each fold's scores by the mean and standard deviation of **that fold's own
validation predictions**. Those come from the fold's model applied to data lying entirely before the
scored slice, so the statistic is available at scoring time and introduces no look-ahead.

In [ ]:
def val_index_for(k):
    """Recompute fold k's validation window exactly as the training loop did."""
    pool = _idx(None, EDGES[k] - GAP)
    return pool[-max(1, int(round(len(pool) * VAL_FRAC))):]


calib, z = [], np.full(len(df), np.nan)
for k in range(1, len(EDGES) - 1):
    tag = f"fold{k}"
    ck = CKPT_DIR / f"finbert_oof_{RUN_TAG}_{tag}.pth"
    if not ck.exists():
        print(f"{tag}: no checkpoint, skipped"); continue

    model = build_model()
    model.load_state_dict(torch.load(ck, map_location=device))
    va = val_index_for(k)
    v = predict(model, va).cpu().numpy()
    mu, sd = float(v.mean()), float(v.std())

    sel = np.flatnonzero(scored_by == tag)
    z[sel] = (scores[sel] - mu) / sd
    calib.append(dict(fold=tag, n_val=len(va), val_mean=round(mu, 5), val_sd=round(sd, 5),
                      raw_mean=round(float(scores[sel].mean()), 5),
                      raw_sd=round(float(scores[sel].std()), 5),
                      z_mean=round(float(z[sel].mean()), 3),
                      z_sd=round(float(z[sel].std()), 3)))
    print(f"{tag}: val mu={mu:+.5f} sd={sd:.5f}  ->  scored slice z-mean {z[sel].mean():+.3f} "
          f"z-sd {z[sel].std():.3f}")
    del model; torch.cuda.empty_cache()

calib = pd.DataFrame(calib)
calib.to_csv(OUT / f"oof_fold_calibration_{RUN_TAG}.csv", index=False)
df["oof_scored_by"] = scored_by
df["predicted_sentiment"] = scores
df["predicted_sentiment_z"] = z
print()
print(calib.to_string(index=False))

## Write the columns

Slice 0 (2019-01 → 2020-01) has no prior data to train on and is written as `NaN` rather than 0.
Zero is a real sentiment value here, so filling would conflate "no news", "neutral news" and
"not available". Drop those rows downstream, or carry an availability indicator — do not fill
them.

In [ ]:
# df kept raw's index through the filtering, so sort_index() restores output.csv's row order
ordered = df.sort_index()
ordered[["timestamp", "title", "source", "predicted_sentiment"]].to_csv(OOF_CSV, index=False)
(ordered[["timestamp", "title", "source", "predicted_sentiment_z"]]
    .rename(columns={"predicted_sentiment_z": "predicted_sentiment"})).to_csv(Z_CSV, index=False)
ordered[["timestamp", "title", "source", "predicted_sentiment", "predicted_sentiment_z",
         "oof_scored_by",
         "volume_surge_price", "price_change_4h"]].to_csv(
    OUT / f"predicted_sentiments_finbert_oof_{RUN_TAG}_annotated.csv", index=False)

print(f"wrote {OOF_CSV}   (raw, {int(ordered['predicted_sentiment'].isna().sum())} NaN from slice 0)")
print(f"wrote {Z_CSV}   <- use this one downstream")
print(f"wrote {OUT / f'predicted_sentiments_finbert_oof_{RUN_TAG}_annotated.csv'}")

## Diagnostics

Two correlation targets, because they answer different questions:

- **`volume_surge_price`** is what the model is trained on, so it measures how well the model learned
  its own task. The published leakage signature lives here (0.4767 in-sample vs 0.0712 out).
- **`price_change_4h`** is what **Table 8 of the paper reports**. The published FinBERT correlation
  of 0.0641 is this column on the test window.

Any `*_z.csv` run found on disk is included, so `tone3` and `cls768` can be compared directly.

In [ ]:
old = (pd.read_csv(OLD_CSV)[["timestamp", "title", "predicted_sentiment"]]
         .rename(columns={"predicted_sentiment": "old"})
         .drop_duplicates(subset=["timestamp", "title"]))
base = df.merge(old, on=["timestamp", "title"], how="left")

cols = {"published (single checkpoint)": base["old"].to_numpy()}
found = sorted(p.name[len("predicted_sentiments_finbert_oof_"):-len("_z.csv")]
               for p in CLEANED.glob("predicted_sentiments_finbert_oof_*_z.csv"))
for tag in sorted(set(found) | {RUN_TAG}):
    f = CLEANED / f"predicted_sentiments_finbert_oof_{tag}_z.csv"
    if tag == RUN_TAG:
        cols[f"OOF {tag} (calibrated)"] = df["predicted_sentiment_z"].to_numpy()
    elif f.exists():
        m = (pd.read_csv(f)[["timestamp", "title", "predicted_sentiment"]]
               .rename(columns={"predicted_sentiment": "_other"})     # df already has that name
               .drop_duplicates(subset=["timestamp", "title"]))
        cols[f"OOF {tag} (calibrated)"] = df.merge(
            m, on=["timestamp", "title"], how="left")["_other"].to_numpy()

BOUND = pd.Timestamp("2024-03-28")
pre, post = (df["dt"] < BOUND).to_numpy(), (df["dt"] >= BOUND).to_numpy()
rows = []
for name, s in cols.items():
    for period, mask in [("train 2019-01 -> 2024-03", pre), ("test  2024-03 -> 2025-03", post)]:
        r = dict(column=name, period=period)
        for tgt in ["volume_surge_price", "price_change_4h"]:
            y = df[tgt].to_numpy()
            k = mask & np.isfinite(s) & np.isfinite(y)
            r[f"r_{tgt.split('_')[0]}"] = round(float(np.corrcoef(s[k], y[k])[0, 1]), 4)
            r["n"] = int(k.sum())
        r["sd"] = round(float(np.nanstd(s[mask])), 5)
        rows.append(r)

final = pd.DataFrame(rows)[["column", "period", "n", "r_volume", "r_price", "sd"]]
final["sd_ratio"] = np.nan
for i in range(0, len(final), 2):
    final.loc[i, "sd_ratio"] = round(final.loc[i, "sd"] / final.loc[i + 1, "sd"], 2)
print(final.to_string(index=False))
final.to_csv(OUT / "oof_vs_insample_comparison.csv", index=False)
print(f"\nr_volume = corr with volume_surge_price (training target)")
print(f"r_price  = corr with price_change_4h (Table 8's target; published FinBERT test value 0.0641)")
print(f"\nwrote {OUT / 'oof_vs_insample_comparison.csv'}")

## Notes and remaining levers

Deliberately not changed here, so that the head comparison stays clean:

1. **Labelled-headline coverage.** `output.csv` holds 20 251 labelled headlines. Recovering more
   would roughly triple every fold; fold 1 — the weakest, at 2 261 training rows — benefits most.
2. **Seed-averaging.** Three seeds per fold, scores averaged, is standard variance reduction on a
   target this noisy.
3. **The label straddles publication**: 39.3 % of its variance is price movement realised before the
   headline appeared. A strictly forward window would make the task coherent.

## Next steps

1. Point `itransformer_models/01_itransformer_training.ipynb` and
   `lstm_models/01_lstm_training.ipynb` at the out-of-fold column, and start the forecasting sample
   at 2020-01 (or carry an availability flag) so the slice-0 `NaN`s are not silently zero-filled.
2. Rerun the four iTransformer feature sets and the LSTM, then the backtests.
3. Finance Qwen is deliberately not re-scored: it never feeds the forecasting pipeline (all
   downstream notebooks read the FinBERT column), and its fine-tuned weights were not retained.
